In [1]:
from google.colab import files
uploaded = files.upload()
from scipy.io import arff
import pandas as pd

# Chargement du fichier ARFF
data = arff.loadarff("Training Dataset.arff")

Saving Training Dataset.arff to Training Dataset.arff


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, learning_curve
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.metrics import classification_report, roc_curve, auc, accuracy_score, precision_recall_curve
from sklearn.metrics import confusion_matrix, average_precision_score, f1_score
from sklearn.feature_selection import RFECV, mutual_info_classif

from sklearn.ensemble import IsolationForest
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline

# Imports supplémentaires pour CNN
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Input, Dropout, concatenate
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

# Set display precision & style
np.set_printoptions(precision=4)
pd.set_option('display.precision', 4)
sns.set_style('whitegrid')
plt.style.use('seaborn-v0_8-pastel')

# ---------------------------
# 0. Chargement du dataset ARFF
# ---------------------------
print("="*80)
print("PHISHING DETECTION AVEC XGBOOST+CNN - ANALYSE AVANCÉE")
print("="*80)

# Fonction pour charger un fichier ARFF en DataFrame pandas
def load_arff(file_path):
    data = []
    attributes = []
    data_section_started = False

    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()

            # Ignorer les lignes vides et les commentaires
            if not line or line.startswith('%'):
                continue

            # Collecter les attributs
            if line.lower().startswith('@attribute'):
                attr_name = line.split()[1].strip()
                attributes.append(attr_name)

            # Détecter le début de la section de données
            elif line.lower().startswith('@data'):
                data_section_started = True

            # Collecter les données
            elif data_section_started:
                values = line.split(',')
                data.append(values)

    # Créer le DataFrame
    df = pd.DataFrame(data, columns=attributes)

    # Convertir les valeurs en numérique (en ignorant les erreurs pour les valeurs non-numériques)
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='ignore')

    return df

# Charger le dataset (vous devrez spécifier le chemin vers votre fichier ARFF)
file_path = 'Training Dataset.arff'  # Modifiez ceci
df = load_arff(file_path)

# Identifier la colonne cible (supposons que c'est la dernière colonne)
target_column = df.columns[-1]

# 1. Vérifier les valeurs uniques dans la colonne cible
print("Valeurs uniques initiales dans la colonne cible:", np.unique(df[target_column]))

# 2. Transformer la colonne cible en classification binaire (0 et 1)
df[target_column] = np.where(df[target_column] > 0, 1, 0)

# 3. Vérifier après transformation
print("Valeurs après transformation:", np.unique(df[target_column]))

print(f"Dimensions du dataset: {df.shape[0]} lignes × {df.shape[1]} colonnes")

# Afin de vous assurer que les valeurs numériques sont correctement interprétées
df = df.astype(float)

# ---------------------------
# 1. Analyse exploratoire des données
# ---------------------------
print("\n" + "="*50)
print("ANALYSE EXPLORATOIRE DES DONNÉES")
print("="*50)

# Statistiques descriptives
print("\nStatistiques descriptives:")
print(df.describe())

# Vérification des valeurs manquantes
print("\nValeurs manquantes par colonne:")
print(df.isnull().sum())

# Distribution de la variable cible
print("\nDistribution de la variable cible:")
target_counts = df[target_column].value_counts()
print(target_counts)
print(f"Pourcentage de classe positive: {target_counts[1] / len(df) * 100:.2f}%")
print(f"Pourcentage de classe négative: {target_counts[0] / len(df) * 100:.2f}%")

# Visualisation de la distribution de la cible
plt.figure(figsize=(8, 6))
sns.countplot(x=df[target_column])
plt.title('Distribution de la variable cible')
plt.xlabel('Classe')
plt.ylabel('Nombre d\'échantillons')
plt.savefig('target_distribution.png')
plt.close()

# ---------------------------
# 2. Préparation des données
# ---------------------------
print("\n" + "="*50)
print("PRÉPARATION DES DONNÉES")
print("="*50)

# Séparation des caractéristiques et de la cible
X = df.drop(target_column, axis=1)
y = df[target_column]

# Division train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Taille de l'ensemble d'entraînement: {X_train.shape[0]} échantillons")
print(f"Taille de l'ensemble de test: {X_test.shape[0]} échantillons")

# Normalisation des données
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Détection et gestion des valeurs aberrantes
print("\nDétection des valeurs aberrantes avec Isolation Forest...")
iso_forest = IsolationForest(contamination=0.05, random_state=42)
outliers = iso_forest.fit_predict(X_train_scaled)
print(f"Nombre de valeurs aberrantes détectées: {(outliers == -1).sum()}")

# Gestion du déséquilibre des classes
print("\nGestion du déséquilibre des classes avec SMOTE...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
print(f"Distribution après rééchantillonnage: {np.bincount(y_train_resampled.astype(int))}")

# Sélection des caractéristiques
print("\nSélection récursive des caractéristiques avec validation croisée...")
xgb_base = xgb.XGBClassifier(random_state=42)
rfecv = RFECV(estimator=xgb_base, step=1, cv=StratifiedKFold(5), scoring='f1', n_jobs=-1)
rfecv.fit(X_train_resampled, y_train_resampled)

print(f"Nombre optimal de caractéristiques: {rfecv.n_features_}")
print(f"Caractéristiques sélectionnées: {X.columns[rfecv.support_].tolist()}")

# Utiliser uniquement les caractéristiques sélectionnées
X_train_selected = X_train_resampled[:, rfecv.support_]
X_test_selected = X_test_scaled[:, rfecv.support_]

# Préparation des données pour CNN
X_train_cnn = X_train_selected.reshape(X_train_selected.shape[0], X_train_selected.shape[1], 1)
X_test_cnn = X_test_selected.reshape(X_test_selected.shape[0], X_test_selected.shape[1], 1)

# Encodage des étiquettes pour le CNN (format catégorique)
y_train_cat = to_categorical(y_train_resampled, num_classes=2)
y_test_cat = to_categorical(y_test, num_classes=2)

# ---------------------------
# 3.1 Modélisation avec XGBoost
# ---------------------------
print("\n" + "="*50)
print("MODÉLISATION AVEC XGBOOST")
print("="*50)

# Création du modèle XGBoost
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    n_estimators=100,
    random_state=42
)

# Grille de paramètres pour l'optimisation
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'n_estimators': [50, 100, 200],
    'min_child_weight': [1, 3, 5]
}

# Optimisation des hyperparamètres avec validation croisée
print("\nOptimisation des hyperparamètres pour XGBoost...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='f1',
    cv=cv,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train_selected, y_train_resampled)

print(f"Meilleurs paramètres pour XGBoost: {grid_search.best_params_}")
print(f"Meilleure performance (F1): {grid_search.best_score_:.4f}")

# Modèle final avec les meilleurs paramètres
best_xgb = grid_search.best_estimator_
best_xgb.fit(X_train_selected, y_train_resampled)

# Importance des caractéristiques
print("\nImportance des caractéristiques (XGBoost):")
feature_names = X.columns[rfecv.support_].tolist()
importance = best_xgb.feature_importances_
indices = np.argsort(importance)[::-1]

for i, idx in enumerate(indices):
    if i < 10:  # Afficher les 10 caractéristiques les plus importantes
        print(f"{feature_names[idx]}: {importance[idx]:.4f}")

# Visualisation de l'importance des caractéristiques
plt.figure(figsize=(12, 8))
xgb.plot_importance(best_xgb, max_num_features=10, height=0.8)
plt.title('Importance des caractéristiques (XGBoost)')
plt.savefig('feature_importance_xgb.png')
plt.close()

# ---------------------------
# 3.2 Modélisation avec CNN
# ---------------------------
print("\n" + "="*50)
print("MODÉLISATION AVEC CNN")
print("="*50)

# Configuration de TensorFlow pour reproduire les résultats
tf.random.set_seed(42)

# Création du modèle CNN
def create_cnn_model(input_shape):
    model = Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Conv1D(filters=32, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Flatten())
    model.add(Dense(100, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(2, activation='softmax'))

    model.compile(loss='categorical_crossentropy',
                  optimizer=Adam(learning_rate=0.001),
                  metrics=['accuracy'])
    return model

# Création et entraînement du modèle CNN
input_shape = (X_train_cnn.shape[1], 1)
cnn_model = create_cnn_model(input_shape)

print(cnn_model.summary())

# Callbacks pour early stopping et sauvegarde du meilleur modèle
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('best_cnn_model.h5', save_best_only=True, monitor='val_accuracy')

# Entraînement du modèle CNN
history = cnn_model.fit(
    X_train_cnn, y_train_cat,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stopping, model_checkpoint],
    verbose=1
)

# Visualisation de l'historique d'entraînement
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Précision du CNN')
plt.xlabel('Époque')
plt.ylabel('Précision')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Perte du CNN')
plt.xlabel('Époque')
plt.ylabel('Perte')
plt.legend()
plt.savefig('cnn_training_history.png')
plt.close()

# ---------------------------
# 3.3 Modèle Hybride XGBoost + CNN
# ---------------------------
print("\n" + "="*50)
print("MODÉLISATION HYBRIDE XGBOOST + CNN")
print("="*50)

# Option 1: Transformer le modèle Sequential en modèle fonctionnel
input_tensor = tf.keras.layers.Input(shape=(X_train_cnn.shape[1], 1))
output_tensor = cnn_model(input_tensor)
functional_model = Model(inputs=input_tensor, outputs=output_tensor)

# Maintenant vous pouvez créer l'extracteur de caractéristiques
feature_extractor = Model(
    inputs=functional_model.input,
    outputs=functional_model.get_layer(index=-2).output
)
X_train_cnn_features = feature_extractor.predict(X_train_cnn)
X_test_cnn_features = feature_extractor.predict(X_test_cnn)

# Probabilités de prédiction du modèle XGBoost
xgb_train_proba = best_xgb.predict_proba(X_train_selected)
xgb_test_proba = best_xgb.predict_proba(X_test_selected)

# Probabilités de prédiction du modèle CNN
cnn_train_proba = cnn_model.predict(X_train_cnn)
cnn_test_proba = cnn_model.predict(X_test_cnn)

# Combinaison des caractéristiques et des probabilités pour l'entrainement du modèle final
if len(X_train_cnn_features.shape) == 3:
    X_train_cnn_features = X_train_cnn_features.reshape(X_train_cnn_features.shape[0], -1)
    print(f"X_train_cnn_features reshaped: {X_train_cnn_features.shape}")

# Combinaison des caractéristiques
X_train_hybrid = np.hstack([
    X_train_selected,                  # Caractéristiques originales
    X_train_cnn_features,              # Caractéristiques extraites du CNN
    xgb_train_proba,                   # Probabilités du XGBoost
    cnn_train_proba                    # Probabilités du CNN
])

print(f"X_train_hybrid shape final: {X_train_hybrid.shape}")

# Assurez-vous que X_test_cnn_features soit 2D et non 3D
if len(X_test_cnn_features.shape) == 3:
    X_test_cnn_features = X_test_cnn_features.reshape(X_test_cnn_features.shape[0], -1)
    print(f"X_test_cnn_features reshaped: {X_test_cnn_features.shape}")

# Combinaison des caractéristiques pour le jeu de test
X_test_hybrid = np.hstack([
    X_test_selected,                   # Caractéristiques originales
    X_test_cnn_features,               # Caractéristiques extraites du CNN
    xgb_test_proba,                    # Probabilités du XGBoost
    cnn_test_proba                     # Probabilités du CNN
])

print(f"X_test_hybrid shape final: {X_test_hybrid.shape}")

# Création du modèle final (XGBoost)
hybrid_model = xgb.XGBClassifier(
    objective='binary:logistic',
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    n_estimators=100,
    random_state=42
)

# Entrainement du modèle hybride
hybrid_model.fit(X_train_hybrid, y_train_resampled)

# ---------------------------
# 4. Évaluation des modèles
# ---------------------------
print("\n" + "="*50)
print("ÉVALUATION DES MODÈLES")
print("="*50)

# Fonction pour évaluer et visualiser les performances d'un modèle
def evaluate_model(model, X_test, y_test, model_name, proba_func=None):
    print(f"\nÉvaluation du modèle {model_name}:")

    if proba_func is None:
        # Pour les modèles sklearn comme XGBoost
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        # Pour les modèles comme CNN qui nécessitent une fonction spéciale
        y_prob = proba_func(X_test)
        y_pred = (y_prob > 0.5).astype(int)

    # Métriques de classification
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    avg_precision = average_precision_score(y_test, y_prob)

    print(f"Précision: {accuracy:.4f}")
    print(f"Score F1: {f1:.4f}")
    print(f"Précision moyenne: {avg_precision:.4f}")

    # Matrice de confusion
    conf_matrix = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-Phishing', 'Phishing'],
                yticklabels=['Non-Phishing', 'Phishing'])
    plt.xlabel('Prédiction')
    plt.ylabel('Réalité')
    plt.title(f'Matrice de confusion ({model_name})')
    plt.savefig(f'confusion_matrix_{model_name}.png')
    plt.close()

    # Rapport de classification détaillé
    print("\nRapport de classification:")
    print(classification_report(y_test, y_pred))

    # Courbe ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Courbe ROC (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Taux de faux positifs')
    plt.ylabel('Taux de vrais positifs')
    plt.title(f'Courbe ROC ({model_name})')
    plt.legend(loc='lower right')
    plt.savefig(f'roc_curve_{model_name}.png')
    plt.close()

    # Courbe Précision-Rappel
    precision, recall, _ = precision_recall_curve(y_test, y_prob)

    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, color='blue', lw=2, label=f'Précision-Rappel (AP = {avg_precision:.4f})')
    plt.axhline(y=sum(y_test)/len(y_test), color='red', linestyle='--', label='Référence')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Rappel')
    plt.ylabel('Précision')
    plt.title(f'Courbe Précision-Rappel ({model_name})')
    plt.legend(loc='lower left')
    plt.savefig(f'precision_recall_curve_{model_name}.png')
    plt.close()

    return accuracy, f1, avg_precision, roc_auc

# Évaluation du modèle XGBoost
xgb_metrics = evaluate_model(best_xgb, X_test_selected, y_test, "XGBoost")

# Évaluation du modèle CNN
cnn_metrics = evaluate_model(None, X_test_cnn, y_test, "CNN",
                             proba_func=lambda x: cnn_model.predict(x)[:, 1])

# Évaluation du modèle hybride
hybrid_metrics = evaluate_model(hybrid_model, X_test_hybrid, y_test, "Hybrid")

# Comparaison des modèles
models = ['XGBoost', 'CNN', 'Hybride XGBoost+CNN']
metrics = [xgb_metrics, cnn_metrics, hybrid_metrics]

metrics_df = pd.DataFrame({
    'Modèle': models,
    'Précision': [m[0] for m in metrics],
    'F1-Score': [m[1] for m in metrics],
    'AP': [m[2] for m in metrics],
    'AUC-ROC': [m[3] for m in metrics]
})

print("\nComparaison des modèles:")
print(metrics_df)

# Visualisation de la comparaison
plt.figure(figsize=(12, 8))
metrics_long = pd.melt(metrics_df, id_vars=['Modèle'],
                       value_vars=['Précision', 'F1-Score', 'AP', 'AUC-ROC'],
                       var_name='Métrique', value_name='Valeur')

sns.barplot(x='Métrique', y='Valeur', hue='Modèle', data=metrics_long)
plt.title('Comparaison des performances des modèles')
plt.ylim(0, 1.0)
plt.savefig('model_comparison.png')
plt.close()

# ---------------------------
# 5. Enregistrement des modèles
# ---------------------------
print("\n" + "="*50)
print("ENREGISTREMENT DES MODÈLES")
print("="*50)

# Sauvegarder le modèle XGBoost
best_xgb.save_model('xgboost_phishing_model.json')
print("Modèle XGBoost enregistré sous 'xgboost_phishing_model.json'")

# Le modèle CNN a déjà été sauvegardé via ModelCheckpoint

# Sauvegarder le modèle hybride final
hybrid_model.save_model('hybrid_phishing_model.json')
print("Modèle hybride enregistré sous 'hybrid_phishing_model.json'")

# Sauvegarde des transformations pour un usage ultérieur
import joblib
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(rfecv, 'feature_selector.pkl')
joblib.dump(feature_extractor, 'cnn_feature_extractor.pkl')
print("Transformations enregistrées (scaler.pkl, feature_selector.pkl, cnn_feature_extractor.pkl)")

# Fonction pour prédire avec le modèle hybride complet
def predict_with_hybrid(data, scaler, rfecv, xgb_model, cnn_feature_extractor, hybrid_model):
    # Prétraitement
    data_scaled = scaler.transform(data)
    data_selected = data_scaled[:, rfecv.support_]

    # Prédictions XGBoost
    xgb_proba = xgb_model.predict_proba(data_selected)

    # Préparation pour CNN
    data_cnn = data_selected.reshape(data_selected.shape[0], data_selected.shape[1], 1)

    # Extraction des caractéristiques CNN
    cnn_features = cnn_feature_extractor.predict(data_cnn)

    # Prédictions CNN
    cnn_proba = cnn_model.predict(data_cnn)

    # Combinaison des caractéristiques
    data_hybrid = np.hstack([
        data_selected,
        cnn_features,
        xgb_proba,
        cnn_proba
    ])

    # Prédiction finale
    return hybrid_model.predict_proba(data_hybrid)[:, 1]

# Exemple de code pour utiliser le modèle hybride
def example_prediction(new_data):
    # Charger les modèles et transformations
    scaler = joblib.load('scaler.pkl')
    rfecv = joblib.load('feature_selector.pkl')
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model('xgboost_phishing_model.json')
    cnn_model = tf.keras.models.load_model('best_cnn_model.h5')
    feature_extractor = joblib.load('cnn_feature_extractor.pkl')
    hybrid_model = xgb.XGBClassifier()
    hybrid_model.load_model('hybrid_phishing_model.json')

    # Prédiction
    proba = predict_with_hybrid(new_data, scaler, rfecv, xgb_model, feature_extractor, hybrid_model)

    return proba > 0.5  # True pour phishing, False sinon

print("\n" + "="*80)
print("ANALYSE TERMINÉE")
print("="*80)

PHISHING DETECTION AVEC XGBOOST+CNN - ANALYSE AVANCÉE
Valeurs uniques initiales dans la colonne cible: [-1  1]
Valeurs après transformation: [0 1]
Dimensions du dataset: 11055 lignes × 31 colonnes

ANALYSE EXPLORATOIRE DES DONNÉES

Statistiques descriptives:
       having_IP_Address  URL_Length  Shortining_Service  having_At_Symbol  \
count         11055.0000  11055.0000          11055.0000        11055.0000   
mean              0.3138     -0.6332              0.7388            0.7006   
std               0.9495      0.7661              0.6740            0.7136   
min              -1.0000     -1.0000             -1.0000           -1.0000   
25%              -1.0000     -1.0000              1.0000            1.0000   
50%               1.0000     -1.0000              1.0000            1.0000   
75%               1.0000     -1.0000              1.0000            1.0000   
max               1.0000      1.0000              1.0000            1.0000   

       double_slash_redirecting  Prefi

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 27, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 13, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 11, 32)         │         6,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 5, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 160)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │        16,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           202 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,734 (88.80 KB)

 Trainable params: 22,734 (88.80 KB)

...

None
Epoch 1/50
277/278 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8396 - loss: 0.3707

278/278 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.8400 - loss: 0.3701 - val_accuracy: 0.9371 - val_loss: 0.1546
Epoch 2/50
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9188 - loss: 0.2036 - val_accuracy: 0.9300 - val_loss: 0.1570
Epoch 3/50
278/278 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9259 - loss: 0.1847 - val_accuracy: 0.9310 - val_loss: 0.1673
Epoch 4/50
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9251 - loss: 0.1782 - val_accuracy: 0.9341 - val_loss: 0.1564
Epoch 5/50
272/278 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9319 - loss: 0.1667

278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9320 - loss: 0.1666 - val_accuracy: 0.9412 - val_loss: 0.1444
Epoch 6/50
274/278 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9346 - loss: 0.1610

278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9346 - loss: 0.1609 - val_accuracy: 0.9422 - val_loss: 0.1293
Epoch 7/50
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9347 - loss: 0.1498 - val_accuracy: 0.9320 - val_loss: 0.1567
Epoch 8/50
278/278 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9371 - loss: 0.1492 - val_accuracy: 0.9361 - val_loss: 0.1433
Epoch 9/50
278/278 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9399 - loss: 0.1445 - val_accuracy: 0.9381 - val_loss: 0.1460
Epoch 10/50
278/278 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9413 - loss: 0.1419 - val_accuracy: 0.9402 - val_loss: 0.1448
Epoch 11/50
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9391 - loss: 0.1414 - val_accuracy: 0.9361 - val_loss: 0.1470
Epoch 12/50
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9431 - loss: 0.1379 - val_accuracy: 0.9422 - val_loss: 0.1397
Epoch 13/50
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9437 - loss: 0.1327 - val_accuracy: 0.9391 -

<Figure size 1200x800 with 0 Axes>